In [1]:
import os
import sys
import numpy as np
import subprocess
from shutil import which

In [ ]:
# 0..21 から 21(=index 20) を除くチーム一覧
# prep: 1-22 (21 is missing)
# contest: 1-20,22-24
def get_teams():
    arr = np.arange(1, 25)
    return np.delete(arr, 20)  # 21 をスキップ（コンテスト仕様に合わせる）

In [3]:
def loop_for_all_teams(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...] のようなリスト
    dry_run: True -> 実行せず展開コマンドのみ表示
    strict: True -> {id:02d} が一つも無ければ例外
    continue_on_error: True -> 失敗しても次の team へ。False -> そこで中断
    cwd: サブプロセスの作業ディレクトリ（attack/ ディレクトリの相対パス解決に使える）
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    # 'python' を実行ファイルに置換（環境ズレ回避）
    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    # 事前: 実行ファイルの存在チェック（python 以外の最初の実体コマンドにも対応）
    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        # 簡易プリフライト: 既知の入力系ファイルっぽい引数を存在確認
        # （.csv, .json かつ -o/--out* ではない位置を対象にする）
        def is_out_flag(i):
            return isinstance(cmd[i-1], str) and (
                cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or cmd[i-1].startswith("--out")
            )

        missing_inputs = []
        for i, a in enumerate(cmd):
            if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                if not is_out_flag(i):  # 出力ではなく入力と推定
                    apath = a if cwd is None else os.path.join(cwd, a)
                    if not os.path.exists(apath):
                        missing_inputs.append(a)

        print(">>", " ".join(cmd))
        if missing_inputs:
            msg = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", msg)
                continue
            else:
                raise FileNotFoundError(msg)

        if dry_run:
            continue

        try:
            # 標準出力・標準エラーを取得して、失敗時に見せる
            completed = subprocess.run(
                cmd, check=True, cwd=cwd,
                capture_output=True, text=True
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
            if e.stdout:
                print("--- stdout ---")
                print(e.stdout.strip())
            if e.stderr:
                print("--- stderr ---")
                print(e.stderr.strip())
            if not continue_on_error:
                raise

In [ ]:
# team=22 のみで実行するバージョン(テスト用)
def loop_for_team_22(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    指定コマンドを team=22 のみで実行・検証する
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    team = 22
    cmd = cmd0[:]
    for ind in id_indices:
        cmd[ind] = cmd[ind].format(id=team)

    def is_out_flag(i):
        return isinstance(cmd[i-1], str) and (
            cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
            or cmd[i-1].startswith("--out")
        )

    missing_inputs = []
    for i, a in enumerate(cmd):
        if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
            if not is_out_flag(i):
                apath = a if cwd is None else os.path.join(cwd, a)
                if not os.path.exists(apath):
                    missing_inputs.append(a)

    print(">>", " ".join(cmd))
    if missing_inputs:
        msg = f"[team {team}] Missing input files: {missing_inputs}"
        if continue_on_error:
            print("!!", msg)
        else:
            raise FileNotFoundError(msg)

    if dry_run:
        return

    try:
        completed = subprocess.run(
            cmd, check=True, cwd=cwd,
            capture_output=True, text=True
        )
        if completed.stdout:
            print(completed.stdout.strip())
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
        if e.stdout:
            print("--- stdout ---")
            print(e.stdout.strip())
        if e.stderr:
            print("--- stderr ---")
            print(e.stderr.strip())
        if not continue_on_error:
            raise

In [6]:
#必要なら cwd='プロジェクトのルート' を指定（例: cwd=r'c:\work\pwscup2025'）
cwd = r"/home/kikuchih/pwscup2025-scripts"  # <- 適宜書き換え
os.chdir(cwd)

In [ ]:
## Create output folder and Place "PWSCUP2025_Pre_Data_for_Attack" files
### create "out/"
### extract PWSCUP2025_Pre_Data_for_Attack_** to out/PWSCUP2025_Pre_Data_for_Attack/
#### ex. out/PWSCUP2025_Pre_Data_for_Attack/A01.csv


In [7]:
## Fix csv files
fix_csv = ["python", "util/check_and_fix_csv.py", "out\PWSCUP2025_Pre_Data_for_Attack\C{id:02d}.csv", "data\pre_columns_range.json", "out\PWSCUP2025_Pre_Data_for_Attack\C{id:02d}_fix.csv"]
loop_for_all_teams(fix_csv)
print(f"csv fix completed")

>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python util/check_and_fix_csv.py out\PWSCUP2025_Pre_Data_for_Attack\C01.csv data\pre_columns_range.json out\PWSCUP2025_Pre_Data_for_Attack\C01_fix.csv


FileNotFoundError: [team 1] Missing input files: ['out\\PWSCUP2025_Pre_Data_for_Attack\\C01.csv', 'data\\pre_columns_range.json', 'out\\PWSCUP2025_Pre_Data_for_Attack\\C01_fix.csv']

In [17]:
## Attack of Ci with original samples
Ci_attack_original = ["python", "attack/attack_Ci.py", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "-o", "out/C{id:02d}_inferred.csv"]
loop_for_all_teams(Ci_attack_original)
print(f"sample Ci-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv -o out/C01_inferred.csv
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv -o out/C02_inferred.csv
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A03.csv out/PWSCUP2025_Pre_Data_for_Attack/C03_fix.csv -o out/C03_inferred.csv
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A04.csv out/PWSCUP2025_Pre_Data_for_Attack/C04_fix.csv -o out/C04_inferred.csv
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A05.csv out/PWSCU

In [18]:
## Attack of Ci with extended version
Ci_attack_extended = ["python", "attack/attack_Ci_ex.py", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "-o", "out/C{id:02d}_inferred_ex.csv", "-k", "1"]
loop_for_all_teams(Ci_attack_extended)
print(f"extended Ci-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv -o out/C01_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv -o out/C02_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A03.csv out/PWSCUP2025_Pre_Data_for_Attack/C03_fix.csv -o out/C03_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A04.csv out/PWSCUP2025_Pre_Data_for_Attack/C04_fix.csv -o out/C04_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex.py out/P

In [19]:
## Attack of Ci with k-NN version
mode = "nn"
k = 5 # choose Nearest k neighbors
Ci_attack_knn = ["python", "attack/attack_Ci_ex_greedy.py", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-m", mode, "-k", str(k), "-o", f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv"]
loop_for_all_teams(Ci_attack_knn)
print(f"k-NN Ci-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/A01.csv -m nn -k 5 -o out/C01_inferred_ex_greedy_k5_nn.csv
dists: [[1.34626579 1.75256181 2.14992976 2.16133809 2.17490435]
 [0.84782469 0.91240346 0.99158037 1.00940716 1.02233338]
 [0.57717073 0.67538285 0.70600224 0.77023667 0.79260063]
 ...
 [1.25265384 1.32750666 1.35667264 1.36075032 1.37555325]
 [0.62510413 0.65466034 0.66586661 0.67827851 0.68271232]
 [0.80501175 0.86199027 0.90041375 1.06054902 1.23266959]]
inds: [[43416 72660 45026 10865 22601]
 [44524 85869 34405 52516 47774]
 [88977 86553 67643 36144 48543]
 ...
 [33283 18698  8797 59885 32396]
 [29881  9209  5322 59978 78543]
 [ 9687 98492 21321 85345  7482]]
dists.shape: (10000, 5), inds.shape: (10000, 5)
1.3462657928466797 43416
idx: [43416 72660 45026 ... 21321 85345  7482]
distances: [1.34626579 1.75256181 2.14992976 ... 0.90041375 1.06054902 1.23266959]
idx

In [20]:
## Attack of Ci with greedy-k-NN version
mode = "greedy"
k = 300 # choose greedy k ranks (need enough big k for computation)
Ci_attack_greedy = ["python", "attack/attack_Ci_ex_greedy.py", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-m", mode, "-k", str(k), "-o", f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv", "--out-map", f"out/C{{id:02d}}_matchmap_k{k}.csv"]
loop_for_all_teams(Ci_attack_greedy)
print(f"greedy-k-NN Ci-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/A01.csv -m greedy -k 300 -o out/C01_inferred_ex_greedy_k300_greedy.csv --out-map out/C01_matchmap_k300.csv
inferred was successfully saved.
match table was successfully saved to out/C01_matchmap_k300.csv
[stats greedy] selected=10000/100000
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/A02.csv -m greedy -k 300 -o out/C02_inferred_ex_greedy_k300_greedy.csv --out-map out/C02_matchmap_k300.csv
inferred was successfully saved.
match table was successfully saved to out/C02_matchmap_k300.csv
[stats greedy] selected=10000/100000
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_ex_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/C03_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/A03.csv -m greedy -k 300 -

In [21]:
## Attack of Di with original samples
### output files will be created at current working directory: inferred_membership1_{id}_ex.csv, inferred_membership2_{id}_ex.csv
Di_attack_original = ["python", "attack/attack_Di.py", "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv"]
loop_for_all_teams(Di_attack_original)
print(f"sample Di-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di.py out/PWSCUP2025_Pre_Data_for_Attack/D01.json out/PWSCUP2025_Pre_Data_for_Attack/A01.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di.py out/PWSCUP2025_Pre_Data_for_Attack/D02.json out/PWSCUP2025_Pre_Data_for_Attack/A02.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di.py out/PWSCUP2025_Pre_Data_for_Attack/D03.json out/PWSCUP2025_Pre_Data_for_Attack/A03.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di.py out/PWSCUP2025_Pre_Data_for_Attack/D04.json out/PWSCUP2025_Pre_Data_for_Attack/A04.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di.py out/PWSCUP2025_Pre_Data_for_

KeyboardInterrupt: 

In [ ]:
## Attack of Di with extended version
# python attack\attack_Di_ex.py out\PWSCUP2025_Pre_Data_for_Attack\D22.json out\PWSCUP2025_Pre_Data_for_Attack\A22.csv [--pred-threshold 0.5] [--pred-topk 10000] [--pred-pos-ratio 0.10] [--conf-threshold 0.1] [--conf-topk 10000] [--conf-pos-ratio 0.10] --out-pred out/inferred_membership1_22_ex.csv --out-conf out/inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
Di_attack_extended = ["python", "attack/attack_Di_ex.py", "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "--out-pred", "out/inferred_membership1_{id:02d}_ex.csv", "--out-conf", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Di_attack_extended)
print(f"extended Di-attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di_ex.py out/PWSCUP2025_Pre_Data_for_Attack/D01.json out/PWSCUP2025_Pre_Data_for_Attack/A01.csv --out-pred out/inferred_membership1_01_ex.csv --out-conf out/inferred_membership2_01_ex.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di_ex.py out/PWSCUP2025_Pre_Data_for_Attack/D02.json out/PWSCUP2025_Pre_Data_for_Attack/A02.csv --out-pred out/inferred_membership1_02_ex.csv --out-conf out/inferred_membership2_02_ex.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Di_ex.py out/PWSCUP2025_Pre_Data_for_Attack/D03.json out/PWSCUP2025_Pre_Data_for_Attack/A03.csv --out-pred out/inferred_membership1_03_ex.csv --out-conf out/inferred_membership2_03_ex.csv
inferred was successfully saved.
inferred was successfully saved.
>> c:\Users\takumi\anaconda3\envs\cosmos

In [23]:
## Original Combination Attack on Ci and Di.
# python attack\attack_Di_ex.py out\PWSCUP2025_Pre_Data_for_Attack\D22.json out\PWSCUP2025_Pre_Data_for_Attack\A22.csv [--pred-threshold 0.5] [--pred-topk 10000] [--pred-pos-ratio 0.10] [--conf-threshold 0.1] [--conf-topk 10000] [--conf-pos-ratio 0.10] --out-pred out/inferred_membership1_22_ex.csv --out-conf out/inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
Combi_attack_original = ["python", "attack/attack_example_ex.py", "--Ai_csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-o", "out/Fij_{id:02d}.csv", "out/C{id:02d}_inferred.csv", "out/inferred_membership1_{id:02d}_ex.csv", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Combi_attack_original)
print(f"original combination attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv out/PWSCUP2025_Pre_Data_for_Attack/A01.csv -o out/Fij_01.csv out/C01_inferred.csv out/inferred_membership1_01_ex.csv out/inferred_membership2_01_ex.csv
[MixAttack] result: selected=66575/100000 (Ci sum=1972, Pred sum=89435, Conf sum=66130)
inferred was successfully saved as out/Fij_01.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv out/PWSCUP2025_Pre_Data_for_Attack/A02.csv -o out/Fij_02.csv out/C02_inferred.csv out/inferred_membership1_02_ex.csv out/inferred_membership2_02_ex.csv
[MixAttack] result: selected=57171/100000 (Ci sum=4683, Pred sum=84662, Conf sum=55870)
inferred was successfully saved as out/Fij_02.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv out/PWSCUP2025_Pre_Data_for_Attack/A03.csv -o out/Fij_03.csv out/C03_inferred.csv out/inferred_membership1_03_ex.csv out/inferred_membership2_03_ex.csv
[

In [24]:
## Extended Combination Attack on Ci and Di.
# python attack\attack_example_ex.py --Ai_csv out\PWSCUP2025_Pre_Data_for_Attack\A22.csv -o out\Fij_22.csv -l 10000 out\C22_inferred.csv out\inferred_membership1_22_ex.csv out\inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
limit = 10000 # limit of number of samples to be inferred

Combi_attack_extended = ["python", "attack/attack_example_ex.py", "--Ai_csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-o", "out/Fij_{id:02d}.csv", "-l", str(limit), f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv", "out/inferred_membership1_{id:02d}_ex.csv", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Combi_attack_extended)
print(f"extended combination attack completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv out/PWSCUP2025_Pre_Data_for_Attack/A01.csv -o out/Fij_01.csv -l 10000 out/C01_inferred_ex_greedy_k300_greedy.csv out/inferred_membership1_01_ex.csv out/inferred_membership2_01_ex.csv
[MixAttack] top-10000 selected (limit=10000)
[MixAttack] result: selected=10000/100000 (Ci sum=10000, Pred sum=89435, Conf sum=66130)
inferred was successfully saved as out/Fij_01.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv out/PWSCUP2025_Pre_Data_for_Attack/A02.csv -o out/Fij_02.csv -l 10000 out/C02_inferred_ex_greedy_k300_greedy.csv out/inferred_membership1_02_ex.csv out/inferred_membership2_02_ex.csv
[MixAttack] top-10000 selected (limit=10000)
[MixAttack] result: selected=10000/100000 (Ci sum=10000, Pred sum=84662, Conf sum=55870)
inferred was successfully saved as out/Fij_02.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_example_ex.py --Ai_csv

In [25]:
# New Di->Ci scoring attack (union of Di selections, rank by Ci distance and |pred - y|)
# Example knobs: threshold-based (pred=0.5, conf=0.3), k=5, select topn=10000
New_DiCi_scoring = [
    "python", "attack/new_attackDi_Ci.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json",
    "--pred-threshold", "0.5",
    "--conf-threshold", "0.25",
    "--mode", "intersection",
    "-k", "5",
    "--w-conf", "1.0",
    # "--auto-wdist",
    "--topn", "10000",
    "-o", "out/Fij_new_{id:02d}.csv",
    "--out-rank", "out/Fij_new_{id:02d}_rank.csv",
]
loop_for_all_teams(New_DiCi_scoring)
print(f"new Di->Ci scoring attack completed")


>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/new_attackDi_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D01.json --pred-threshold 0.5 --conf-threshold 0.25 --mode intersection -k 5 --w-conf 1.0 --topn 10000 -o out/Fij_new_01.csv --out-rank out/Fij_new_01_rank.csv
[NewAttackDiCi] selected=10000/100000 (topn=10000, candidates=78634)
inferred was successfully saved as out/Fij_new_01.csv
rank table was successfully saved as out/Fij_new_01_rank.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/new_attackDi_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D02.json --pred-threshold 0.5 --conf-threshold 0.25 --mode intersection -k 5 --w-conf 1.0 --topn 10000 -o out/Fij_new_02.csv --out-rank out/Fij_new_02_rank.csv
[NewAttackDiCi] selected=10000/100000 (topn=10000, candidates=70866)
inferred was successf

KeyboardInterrupt: 

In [26]:
# New Di->Ci scoring attack (greedy Ci matching; k ignored, ranks expand automatically)
New_DiCi_scoring_greedy = [
    "python", "attack/new_attackDi_Ci_greedy.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json",
    "--pred-threshold", "0.5",
    "--conf-threshold", "0.25",
    "--mode", "intersection",
    "--w-conf", "1.0",
    # "--auto-wdist",
    "--topn", "10000",
    "-o", "out/Fij_new_greedy_{id:02d}.csv",
    "--out-rank", "out/Fij_new_greedy_{id:02d}_rank.csv",
    "--out-map", "out/C{id:02d}_matchmap_greedy.csv",
]
loop_for_all_teams(New_DiCi_scoring_greedy)
print(f"new Di->Ci scoring (greedy) completed")


>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/new_attackDi_Ci_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D01.json --pred-threshold 0.5 --conf-threshold 0.25 --mode intersection --w-conf 1.0 --topn 10000 -o out/Fij_new_greedy_01.csv --out-rank out/Fij_new_greedy_01_rank.csv --out-map out/C01_matchmap_greedy.csv
[NewAttackDiCiGreedy] selected=10000/100000 (topn=10000, candidates=78634)
inferred was successfully saved as out/Fij_new_greedy_01.csv
rank table was successfully saved as out/Fij_new_greedy_01_rank.csv
match table was successfully saved as out/C01_matchmap_greedy.csv
[stats greedy] matched within candidates = 7481/78634 shown in rank table
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/new_attackDi_Ci_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D02.json --pred-threshold 0.5

In [27]:
# Independent Ci+Di (greedy Ci distance + Di |pred - y|)
Ci_Di_independent = [
    "python", "attack/attack_Ci_Di_independent.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json",
    "--w-conf", "1.0",
    # "--auto-wdist",
    "--k-hint", "300",
    "--topn", "10000",
    "-o", "out/Fij_independent_{id:02d}.csv",
    "--out-rank", "out/Fij_independent_{id:02d}_rank.csv",
    "--out-map", "out/C{id:02d}_matchmap_independent.csv",
]
loop_for_all_teams(Ci_Di_independent)
print(f"independent Ci+Di (greedy) scoring completed")

>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_Di_independent.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D01.json --w-conf 1.0 --k-hint 300 --topn 10000 -o out/Fij_independent_01.csv --out-rank out/Fij_independent_01_rank.csv --out-map out/C01_matchmap_independent.csv
[AttackCiDiIndependent] selected=10000/100000 (topn=10000)
inferred was successfully saved as out/Fij_independent_01.csv
rank table was successfully saved as out/Fij_independent_01_rank.csv
match table was successfully saved as out/C01_matchmap_independent.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_Di_independent.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D02.json --w-conf 1.0 --k-hint 300 --topn 10000 -o out/Fij_independent_02.csv --out-rank out/Fij_independent_02_rank.csv --out-map out/C02_matchmap_ind

In [28]:
# Attack of Ci with Hungarian assignment (min-sum matching)
hung_mode = "auto"  # use 'auto' to try full when feasible
k_hung = 300          # number of Ai candidates per Ci in knn mode
Ci_attack_hungarian = [
    "python", "attack/attack_Ci_hungarian.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "-m", hung_mode,
    "-k", str(k_hung),
    "-o", f"out/C{{id:02d}}_inferred_hungarian_{hung_mode}_k{k_hung}.csv",
    "--out-map", f"out/C{{id:02d}}_matchmap_hungarian_{hung_mode}_k{k_hung}.csv",
]
loop_for_all_teams(Ci_attack_hungarian)
print(f"Hungarian Ci-attack completed")


>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv -m auto -k 300 -o out/C01_inferred_hungarian_auto_k300.csv --out-map out/C01_matchmap_hungarian_auto_k300.csv
[AttackCiHungarian] matched=10000 of Ci=10000; selected Ai=10000/100000
inferred was successfully saved as out/C01_inferred_hungarian_auto_k300.csv
match table was successfully saved as out/C01_matchmap_hungarian_auto_k300.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_Ci_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv -m auto -k 300 -o out/C02_inferred_hungarian_auto_k300.csv --out-map out/C02_matchmap_hungarian_auto_k300.csv
[AttackCiHungarian] matched=10000 of Ci=10000; selected Ai=10000/100000
inferred was successfully saved as out/C02_inferred_hungarian_auto_k300.csv
match table was successfully saved as out/C02_matchmap_hungar

In [29]:
# New Di->Ci scoring attack (Hungarian Ci matching; rank Di candidates by matched distance and |pred - y|)
# Example knobs: threshold-based selection, Hungarian knn mode k=300, select top 10,000
New_DiCi_hungarian = [
    "python", "attack/attackDi_Ci_hungarian.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json",
    "--pred-threshold", "0.5",
    "--conf-threshold", "0.25",
    "--mode", "intersection",
    "--hung-mode", "auto",
    "-k", "300",
    "--w-conf", "1.0",
    # "--auto-wdist",
    "--topn", "10000",
    "-o", "out/Fij_new_hung_{id:02d}.csv",
    "--out-rank", "out/Fij_new_hung_{id:02d}_rank.csv",
    "--out-map", "out/C{id:02d}_matchmap_hungarian_used.csv",
]
loop_for_all_teams(New_DiCi_hungarian)
print(f"new Di->Ci Hungarian scoring attack completed")


>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attackDi_Ci_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D01.json --pred-threshold 0.5 --conf-threshold 0.25 --mode intersection --hung-mode auto -k 300 --w-conf 1.0 --topn 10000 -o out/Fij_new_hung_01.csv --out-rank out/Fij_new_hung_01_rank.csv --out-map out/C01_matchmap_hungarian_used.csv
[AttackCiHungarian] matched=10000 of Ci=10000; selected Ai=10000/100000
[AttackDiCiHungarian] selected=10000/100000 (topn=10000, candidates=78634)
inferred was successfully saved as out/Fij_new_hung_01.csv
rank table was successfully saved as out/Fij_new_hung_01_rank.csv
match table was successfully saved as out/C01_matchmap_hungarian_used.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attackDi_Ci_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D

In [ ]:
# AllCi + AllDi (Hungarian): rank by [Hungarian Ci distance + Di |pred - y|], pick top 10,000
AllCi_AllDi_Hungarian = [
    "python", "attack/attack_allCi_allDi_hungarian.py",
    "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv",
    "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json",
    "--hung-mode", "auto",
    "-k", "300",
    "--w-dist", "1.0",
    "--w-conf", "1.0",
    # "--auto-wdist",
    "--topn", "10000",
    "-o", "out/Fij_all_hungarian_{id:02d}.csv",
    "--out-rank", "out/Fij_all_hungarian_{id:02d}_rank.csv",
    "--out-map", "out/C{id:02d}_matchmap_all_hungarian.csv",
]
loop_for_all_teams(AllCi_AllDi_Hungarian)
print(f"AllCi+AllDi (Hungarian) scoring completed")


>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_allCi_allDi_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D01.json --hung-mode auto -k 300 --w-dist 1.0 --w-conf 1.0 --topn 10000 -o out/Fij_all_hungarian_01.csv --out-rank out/Fij_all_hungarian_01_rank.csv --out-map out/C01_matchmap_all_hungarian.csv
[AttackAllCiAllDiHungarian] matched=10000 (capped=10000); selected Ai=10000/100000
inferred was successfully saved as out/Fij_all_hungarian_01.csv
rank table was successfully saved as out/Fij_all_hungarian_01_rank.csv
match table was successfully saved as out/C01_matchmap_all_hungarian.csv
>> c:\Users\takumi\anaconda3\envs\cosmos\python.exe attack/attack_allCi_allDi_hungarian.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/D02.json --hung-mode auto -k 300 --w-dist 1.0 --w-conf 1.0 --topn 10000 -o out/Fij_

In [ ]:
# Creating answer files
## python evaluation\gen_ans.py out\PWSCUP2025_Pre_Data_for_Attack\A22.csv in\B22_3.csv -o out/Z22.csv

In [ ]:
# Evaluation of attack result
# python evaluation\check_ans.py out/Fij_all_hungarian_22.csv out/Z22.csv